In [ ]:
import os
import torch
from botorch import fit_gpytorch_mll
import botorch_tool
import copy
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.transforms import unnormalize, normalize, standardize
import numpy as np
import pandas as pd
import json
from matplotlib import pyplot as plt
import pickle
from botorch.utils.transforms import standardize
from botorch.models.transforms.outcome import Standardize
from copy import copy

torch.set_default_dtype(torch.float64)

tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

BATCH_NUM = 10

In [ ]:
exp_data_in = []
for BATCH_NUM_tmp in range(BATCH_NUM + 1):
    with open(f"./all_experiment_{BATCH_NUM_tmp}.json", "r") as f:
        exp_data_in.append(pd.DataFrame(json.load(f)))

exp_data_in = (
    pd.concat(exp_data_in)
    .sort_values(by=["substance1", "substance2", "volume1"])
    .reset_index(drop=True)
)

if not (exp_data_in.index.astype("int") == np.arange(len(exp_data_in))).all():
    raise ValueError("the index of the expeirments is not right")

# add noise from Pt experiments
exp_data_in["variance_S"] = exp_data_in["variance_S"] + 1.6
exp_data_in["variance_current"] = exp_data_in["variance_current"] + 198.53
exp_data_in
